In [1]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [9]:
MODEL = "gpt-4.1-nano"
db_name = "vector_db_for_dall"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


In [3]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("DALL_FULL_MD_Dataset/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    print(f"Loading documents from {doc_type} folder")
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    print(folder_docs)
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loading documents from company folder
[Document(metadata={'source': 'DALL_FULL_MD_Dataset\\company\\contact.md'}, page_content='DALL Technologies Pvt. Ltd.\nPhone: +91-80-4567-8900\nEmail: contact@dall.com\nSupport: support@dall.com\n')]
Loading documents from contracts folder
[Document(metadata={'source': 'DALL_FULL_MD_Dataset\\contracts\\contract_001_enterprise.md'}, page_content='Enterprise Supply Contract\nDuration: 3 Years\n'), Document(metadata={'source': 'DALL_FULL_MD_Dataset\\contracts\\contract_002_government.md'}, page_content='Government Infrastructure Contract\nDuration: 5 Years\n'), Document(metadata={'source': 'DALL_FULL_MD_Dataset\\contracts\\contract_003_datacenter.md'}, page_content='Data Center Partnership Contract\n'), Document(metadata={'source': 'DALL_FULL_MD_Dataset\\contracts\\contract_004_maintenance.md'}, page_content='Annual Maintenance Contract\n'), Document(metadata={'source': 'DALL_FULL_MD_Dataset\\contracts\\contract_005_cloud.md'}, page_content='Cloud Pro

In [5]:
# encoding = tiktoken.encoding_for_model(MODEL)
# tokens = encoding.encode(documents)
# token_count = len(tokens)
# print(f"Total tokens for {MODEL}: {token_count:,}")


# it will not work because token encode will take only strins but we give document type as input 


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter= RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")


Divided into 26 chunks
First chunk:

page_content='DALL Technologies Pvt. Ltd.
Phone: +91-80-4567-8900
Email: contact@dall.com
Support: support@dall.com' metadata={'source': 'DALL_FULL_MD_Dataset\\company\\contact.md', 'doc_type': 'company'}


# next is to create vector and put the vectors in vector database

In [8]:
embedding =HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


In [11]:
vectorstore=Chroma.from_documents(chunks, embedding, persist_directory=db_name)

# now we need to craete two langchain abstraction 1. llm 2. Retriver

In [ ]:
from langchain_openai import ChatOpenAI

retriever=vectorstore.as_retriever()
llm=ChatOpenAI(model=MODEL, temperature=1)

In [18]:
x=retriever.invoke("What is the DALL-E 3 knowledge base about?")
print("\n".join(doc.page_content for doc in x))

DALL Technologies Pvt. Ltd.
Phone: +91-80-4567-8900
Email: contact@dall.com
Support: support@dall.com
# Employee Profile: John Doe
Role: Senior Hardware Engineer
Skills: Server design, DDR5, Signal integrity, Thermal optimization
Career: Joined DALL in 2019, promoted to Senior Engineer in 2022
DALL Server X100
Price: ₹6.5L–₹9.2L
Features: IPMI, Hot-swap PSU, Secure BIOS
# Employee Profile: Priya Sharma
Role: Memory Systems Engineer
Skills: DDR5, HBM, Firmware validation, Python testing
Career: Joined in 2020, key contributor to HBM stack


## Now we will create one Rag pipeline call to retrieve the data from vector store


In [23]:
system_message_template = """You are a helpful assistant for answering questions about the DALL Company, 
you will use the retrieved information to answer the user's questions accurately and concisely. 
If the retrieved information does not contain the answer, you will say you don't know. 
Here is the retrieved information: {content}"""

In [36]:
from langchain_core.messages import SystemMessage, HumanMessage
def output(input, history):
    retrieved_docs = retriever.invoke(input)
    retrive_data="\n\n".join(doc.page_content for doc in retrieved_docs)
    system_message=system_message_template.format(content= retrive_data)
    response = llm.invoke([SystemMessage(content=system_message), HumanMessage(content=input)])
    return response.content

In [37]:
output("What is DALL",[])

'DALL is a company involved in technology and hardware solutions. For more details, you can contact them at +91-80-4567-8900 or email contact@dall.com.'

In [38]:
import gradio as gr 

In [39]:
gr.ChatInterface(output, title="DALL-E 3 Knowledge Base Chatbot", description="Ask questions about the DALL-E 3 knowledge base.").launch(inbrowser=True)

c:\Users\asus\projects\github_push_llm\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
